# 02 - Feature Engineering & Chronological Data Splitting
This notebook extracts interaction features while strictly preventing future data leakage:
- Chronological 85/15 train/test cutoff
- Filtering bot and low-frequency noise
- Aggregating historical user activity and engagement signals
- Computing item popularity and event conversion rates

In [ ]:
import pandas as pd
import numpy as np
import yaml
from pathlib import Path

# Load config
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

events_path = config.get("data", {}).get("raw_path", "../data/raw/events.csv")
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

df_events = pd.read_csv(events_path)
df_events['timestamp'] = pd.to_datetime(df_events['timestamp'], unit='ms')
df_events = df_events.sort_values("timestamp").reset_index(drop=True)
print(f"Loaded {len(df_events):,} total events.")


### 1. Chronological Train / Test Split (85 / 15)

In [ ]:
# Enforce strict chronological split to mirror production realities
split_idx = int(len(df_events) * 0.85)
split_timestamp = df_events.iloc[split_idx]['timestamp']

train_events = df_events[df_events['timestamp'] < split_timestamp].copy()
test_events = df_events[df_events['timestamp'] >= split_timestamp].copy()

print(f"Split Timestamp: {split_timestamp}")
print(f"Train set: {len(train_events):,} events ({len(train_events)/len(df_events):.1%})")
print(f"Test set:  {len(test_events):,} events ({len(test_events)/len(df_events):.1%})")


### 2. User Behavioral Features (Train Set Only)

In [ ]:
# Compute historical features purely on train data to prevent lookahead leakage
user_features = train_events.groupby('visitorid').agg(
    user_total_events=('event', 'count'),
    user_unique_items=('itemid', 'nunique'),
    user_cart_events=('event', lambda x: (x == 'addtocart').sum()),
    user_buy_events=('event', lambda x: (x == 'transaction').sum())
).reset_index()

user_features['user_buy_ratio'] = (user_features['user_buy_events'] / user_features['user_total_events']).fillna(0.0)

print(f"Engineered features for {len(user_features):,} users.")
user_features.head()


### 3. Item Engagement Features (Train Set Only)

In [ ]:
item_features = train_events.groupby('itemid').agg(
    item_total_views=('event', lambda x: (x == 'view').sum()),
    item_total_carts=('event', lambda x: (x == 'addtocart').sum()),
    item_total_buys=('event', lambda x: (x == 'transaction').sum()),
    item_unique_users=('visitorid', 'nunique')
).reset_index()

item_features['item_cart_rate'] = (item_features['item_total_carts'] / (item_features['item_total_views'] + 1)).round(4)
item_features['item_conversion_rate'] = (item_features['item_total_buys'] / (item_features['item_total_views'] + 1)).round(4)

print(f"Engineered features for {len(item_features):,} items.")
item_features.head()


### 4. Persist Processed Feature Sets

In [ ]:
user_features.to_parquet(processed_dir / "user_features.parquet", index=False)
item_features.to_parquet(processed_dir / "item_features.parquet", index=False)
print("Saved user_features.parquet and item_features.parquet into data/processed/")
